# [SK 07 - AI Foundry Agents with Semantic Kernel](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python)
**Note**: `azure-ai-agents==1.1.0b4` and `azure-ai-projects==1.0.0` are automatically installed by semantic kernel 1.35.0.<br/>

An AzureAIAgent is a specialized agent within the Semantic Kernel framework, designed to provide advanced conversational capabilities with seamless tool integration. It automates tool calling, eliminating the need for manual parsing and invocation. The agent also securely manages conversation history using threads, reducing the overhead of maintaining state. Additionally, the AzureAIAgent supports a variety of built-in tools, including file retrieval, code execution, and data interaction via Bing, Azure AI Search, Azure Functions, and OpenAPI.

To use an AzureAIAgent, an Azure AI Foundry Project must be utilized. 

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name        = "sk_aifoundry_agent-PYTHON"
instructions      = "you are a clever agent"
file_to_search_in = "./data/product_info_1.md"

project_endpoint = os.environ["AIF_BAS_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# 1. Create AI Foundry `AIProjectClient` using [`AzureAIAgent`](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.<br/>
The agent leverages an AzureAIAgentModel configuration and can optionally override default parameters such as temperature, maximum tokens, or instructions.<br/>
Initialize an AzureAIAgent service by providing at minimum an AIProjectClient and an AzureAIAgentModel

In [2]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

# 2. Setting up Resources: `AzureOpenAISettings`
Now that we have the project client created, the call to AzureOpenAISettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [3]:
from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings

azureopenai_settings = AzureOpenAISettings()
azureopenai_settings

AzureOpenAISettings(env_file_path=None, env_file_encoding='utf-8', chat_deployment_name='gpt-4o', responses_deployment_name=None, text_deployment_name='gpt-35-turbo-instruct', embedding_deployment_name='text-embedding-ada-002', text_to_image_deployment_name=None, audio_to_text_deployment_name=None, text_to_audio_deployment_name=None, realtime_deployment_name=None, endpoint=AnyUrl('https://mmoaiswc-01.openai.azure.com/'), base_url=None, api_key=SecretStr('**********'), api_version='2025-04-01-preview', token_endpoint='https://cognitiveservices.azure.com/.default')

# 3. Create CodeInterpreter tool and resources
Upload files into the assistant and retrieve code interpreter tool and resources.

In [4]:
from semantic_kernel.agents import AzureAssistantAgent

# Upload the files to the client
file_ids: list[str] = []
uploaded_file = ""
for path in [file_to_search_in]:
    with open(path, "rb") as file:
        uploaded_file = await project_client.agents.files.upload(file=file, purpose="assistants")
        
        file_ids.append(uploaded_file.id)
        print(f"File {path} uploaded as {uploaded_file.id}\n")

# Get the code interpreter tool and resources
code_interpreter_tools, code_interpreter_tool_resources = AzureAssistantAgent.configure_code_interpreter_tool(file_ids = file_ids)

print(f"code_interpreter_tools: {code_interpreter_tools}\ncode_interpreter_tool_resources: {code_interpreter_tool_resources}")

File ./data/product_info_1.md uploaded as assistant-EEGNHdBYydr6sbSeG3UJVk

code_interpreter_tools: [{'type': 'code_interpreter'}]
code_interpreter_tool_resources: {'code_interpreter': {'file_ids': ['assistant-EEGNHdBYydr6sbSeG3UJVk']}}


# 4. Creating the AI Foundry Agent, e.g. `Agent definition` for the SK agent
Notes:
- This command actually loads / creates the **AI Foundry** `Agent` in the AI Foundry service.
- However, this is still **not** a `Semantic Kernel` Agent, but will be used as a `definition` to create it.
- This assistant still does **not** contain the `plugin` (we could do it, but we'll do it later).

In [5]:
agent_id = "" # for ex: "asst_qLugGQ3nZ0wdWgqS6SwZHGVg"

if agent_id != "":
    aifoundry_agent = await project_client.agents.get_agent(agent_id=agent_id)
else:
    aifoundry_agent = await project_client.agents.create_agent(
        model=azureopenai_settings.chat_deployment_name,
        name=agent_name,
        instructions=instructions,
        tools=code_interpreter_tools,
        tool_resources=code_interpreter_tool_resources,
    )
    
aifoundry_agent

{'id': 'asst_rkPAXsj37bzdTfncqhV2fpj3', 'object': 'assistant', 'created_at': 1754812045, 'name': 'sk_aifoundry_agent-PYTHON', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [{'type': 'code_interpreter'}], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {'code_interpreter': {'file_ids': ['assistant-EEGNHdBYydr6sbSeG3UJVk']}}, 'metadata': {}, 'response_format': 'auto'}

# 5. Creating an SK Agent based on the AI Foundry definition
As we can see, this agent already ontains the associations with CodeInterpreter and Resources (=files to search in)

In [6]:
agent = AzureAIAgent(
    client=project_client,
    definition=aifoundry_agent,
    # I will add the plugin later, otherwise if I have it ready now, I could use `plugins=[LightsPlugin()]`
)

agent

AzureAIAgent(arguments=None, description=None, id='asst_rkPAXsj37bzdTfncqhV2fpj3', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000016DEA75D6A0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='sk_aifoundry_agent-PYTHON', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x0000016DE9EBEF90>, definition={'id': 'asst_rkPAXsj37bzdTfncqhV2fpj3', 'object': 'assistant', 'created_at': 1754812045, 'name': 'sk_aifoundry_agent-PYTHON', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [{'type': 'code_interpreter'}], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {'code_interpreter': {'file_ids': ['assistant-EEGNHdBYydr6sbSeG3UJVk']}}, 'metadata': {}, 'response_format': 'auto'}, polling_opti

# 6. Define native plugin and planner
Here we do the following:
- 6.1 Plugin definition
- 6.2 Plugin association with the SK Agent created in the previous step
- 6.3 Planner settings definition and association with the SK agent for automatic plugin invokation

## 6.1 Plugin definition

In [7]:
class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

## 6.2 Plugin association with the SK Agent created in the previous step
Now, the SK agent does contain the plugin association. 

In [8]:
agent.kernel.add_plugin(LightsPlugin())
agent

AzureAIAgent(arguments=None, description=None, id='asst_rkPAXsj37bzdTfncqhV2fpj3', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000016DEA75D6A0>, plugins={'LightsPlugin': KernelPlugin(name='LightsPlugin', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='LightsPlugin', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False, is_a

## 6.3 Planner settings definition and association with the SK agent for [`plugin invokation`](https://learn.microsoft.com/en-us/semantic-kernel/concepts/ai-services/chat-completion/function-calling/function-invocation)
**Note**: this pattern is used in .NET, it works with Python and ChatCompletion. But it seems not woriking with Python and AI Foundry Agents in SK.

In [9]:
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import AzureChatPromptExecutionSettings

arguments = KernelArguments(
    settings=AzureChatPromptExecutionSettings(
        # By default, functions are set to be automatically invoked.
        # If you want to explicitly enable this behavior, you can do so with the following code:
        # function_choice_behavior=FunctionChoiceBehavior.Auto(auto_invoke=True),
        function_choice_behavior=FunctionChoiceBehavior.NoneInvoke(
            auto_invoke=False, enable_kernel_functions=False), # Auto(), Required() or NoneInvoke()
    )
)

agent.arguments=KernelArguments(settings=arguments.execution_settings)
agent.arguments.execution_settings

{'default': AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=False, maximum_auto_invoke_attempts=0, filters=None, type_=<FunctionChoiceType.NONE: 'none'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, parallel_tool_calls=None, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, max_completion_tokens=None, reasoning_effort=None, extra_body=None)}

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [10]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Hello", 
    "Please toggle the porch light", 
    "What's the status of all lights?", 
    "Thank you",
]

thread: AzureAIAgentThread = None

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        print(f"Message {i} from {AuthorRole.USER}: '{user_input}'")
        # response = await agent.get_response(messages=user_input, thread=thread, arguments=arguments)
        responses = agent.invoke(messages=user_input, thread=thread, arguments=arguments)
        async for response in responses:
            print(f"Message {i} from {AuthorRole.ASSISTANT}: '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

Message 1 from AuthorRole.USER: 'Hello'
Message 1 from AuthorRole.ASSISTANT: 'Hello! How can I assist you today?'

Message 2 from AuthorRole.USER: 'Please toggle the porch light'
Message 2 from AuthorRole.ASSISTANT: 'The porch light has been toggled on successfully! Let me know if you need anything else.'

Message 3 from AuthorRole.USER: 'What's the status of all lights?'
Message 3 from AuthorRole.ASSISTANT: 'Here is the status of all lights:

- **Table Lamp**: Off
- **Porch light**: On
- **Chandelier**: Off

Let me know if you'd like to change the status of any light!'

Message 4 from AuthorRole.USER: 'Thank you'
Message 4 from AuthorRole.ASSISTANT: 'You're welcome! Let me know if there's anything else I can help with. Have a great day! 😊'


Thread <thread_dKyYJASaMU1ikjTqUOkqRL0q> was created to manage the conversation


# Teardown

In [11]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

2 files will now be deleted:
- File 1 of 2: product_info_1.md (id=assistant-EEGNHdBYydr6sbSeG3UJVk) is being deleted...
- File 2 of 2: product_info_1.md (id=assistant-3gLv5QvULmMQNDX47j93Q3) is being deleted...


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [12]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_dKyYJASaMU1ikjTqUOkqRL0q> is being deleted...
2 - Thread <thread_L904oHKoPJJotCO0p52pDuiw> is being deleted...
3 - Thread <thread_i1nJu02VpO7iq81G2FWTG5gl> is being deleted...
4 - Thread <thread_8KlmqMx94axiXhKaU2CFePrv> is being deleted...
5 - Thread <thread_SGZ5pAHwHsrGTGuqfHQTn9s3> is being deleted...
6 - Thread <thread_ZWS8PdswdxU6V3ZbxoqpDS0p> is being deleted...
7 - Thread <thread_pIX7h6DlTMAkmuZcOTEiRZrQ> is being deleted...
8 - Thread <thread_Vs1oSnsxd6wrpu1LwLw3zgGF> is being deleted...
9 - Thread <thread_GBVq9MBNPtp2bblxHb9GtTTz> is being deleted...
10 - Thread <thread_eCnRKsuwsp966JbSAbEYGvjp> is being deleted...
11 - Thread <thread_LpRXuhAWLYdRmRkg5vE53fYP> is being deleted...
12 - Thread <thread_X7uOKnBoHbxVdt1Na8xwcxcS> is being deleted...
13 - Thread <thread_mKt0paoN1zWJYFAtmiBQi5mt> is being deleted...


In [13]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_rkPAXsj37bzdTfncqhV2fpj3> is being deleted...
2 - Agent <asst_KYyeU1Y0lt4QtOEgyVXsOVeg> is being deleted...
